# Task

In [6]:
import pandas as pd
import numpy as np
import random
from sklearn import metrics
from xgboost import XGBClassifier
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, mean_squared_error, mean_absolute_error, r2_score

In [7]:
data = pd.read_csv('data/other/loan_data_exam.csv')
data

,Unnamed: 0,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,40997,35.0,female,High School,157123.0,9,MORTGAGE,6657.0,HOMEIMPROVEMENT,17.37,0.04,5.0,659.0,Yes,0.0
1,44208,26.0,male,Master,42906.0,3,RENT,12958.0,MEDICAL,11.59,0.30,3.0,645.0,No,1.0
2,9915,22.0,NaN,Associate,58958.0,3,RENT,9000.0,DEBTCONSOLIDATION,7.88,0.15,2.0,652.0,Yes,0.0
3,17597,22.0,male,High School,82024.0,0,RENT,7500.0,VENTURE,9.76,0.09,3.0,566.0,Yes,0.0
4,21842,31.0,female,Master,78532.0,8,RENT,6000.0,VENTURE,6.99,0.08,10.0,703.0,No,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,3451,26.0,male,Associate,44825.0,0,MORTGAGE,5000.0,MEDICAL,6.03,0.11,2.0,599.0,No,0.0
9996,24181,30.0,female,Associate,55089.0,5,RENT,10000.0,PERSONAL,7.51,0.18,7.0,NaN,No,0.0
9997,3909,23.0,male,High School,50147.0,1,RENT,4000.0,MEDICAL,7.68,0.08,4.0,672.0,No,0.0
9998,2030,NaN,NaN,Associate,27735.0,0,RENT,2200.0,MEDICAL,11.01,0.08,3.0,588.0,Yes,0.0


In [9]:
def drop_data(data:pd.DataFrame, columns:list):
  data_copy = data.copy()
  data_copy.drop(columns, axis=1, inplace=True)
  return data_copy

In [10]:
def drop_missing_data(data:pd.DataFrame, columns:list):
  data_copy = data.copy()
  data_copy.dropna(subset=columns, inplace=True)
  return data_copy

In [11]:
def simple_impute_data(data:pd.DataFrame, columns:list, strategy:str):
  imputer = SimpleImputer(strategy=strategy)
  data_copy = data.copy()

  for column in columns:
    data_copy[column] = imputer.fit_transform(data_copy[[column]])
  return data_copy

In [12]:
def label_data(data:pd.DataFrame, columns:list):
  encoder = LabelEncoder()
  data_copy = data.copy()

  for column in columns:
    data_copy[column] = encoder.fit_transform(data_copy[[column]].astype(str).values.ravel())

    if 'nan' in encoder.classes_:
      data_copy.loc[data_copy[column] == -1, column] = np.nan
  return data_copy

In [13]:
def knn_impute_data(data:pd.DataFrame, columns:list, n_neighbors:int, weights:str):
  imputer = KNNImputer(n_neighbors=n_neighbors, weights=weights)
  data_copy = data.copy()

  for column in columns:
    data_copy[column] = imputer.fit_transform(data_copy[column].to_numpy().reshape(-1, 1))
  return data_copy

In [14]:
data.sample(10)

,Unnamed: 0,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
9730,2134,22.0,female,High School,45803.0,0,RENT,2400.0,PERSONAL,10.65,0.05,3.0,671.0,No,1.0
772,24808,27.0,female,NaN,92341.0,3,MORTGAGE,10000.0,DEBTCONSOLIDATION,10.99,0.11,9.0,507.0,Yes,0.0
3726,30123,36.0,female,Master,36783.0,15,RENT,5500.0,EDUCATION,8.90,0.15,11.0,739.0,Yes,0.0
9129,26238,29.0,male,Bachelor,72756.0,7,RENT,14000.0,MEDICAL,7.51,0.19,8.0,615.0,Yes,0.0
1869,3904,NaN,female,Associate,48807.0,0,RENT,4000.0,PERSONAL,11.49,0.08,4.0,683.0,No,0.0
7879,31589,36.0,female,High School,51323.0,13,RENT,2000.0,VENTURE,11.14,0.04,11.0,580.0,Yes,0.0
1249,28671,32.0,male,Associate,46778.0,12,RENT,6350.0,DEBTCONSOLIDATION,5.79,0.14,8.0,661.0,Yes,0.0
312,30466,48.0,female,High School,66983.0,27,MORTGAGE,10000.0,HOMEIMPROVEMENT,11.01,0.15,11.0,734.0,No,0.0
8296,25060,34.0,male,Master,37048.0,10,RENT,12000.0,PERSONAL,11.01,0.32,6.0,727.0,No,1.0
8833,5828,25.0,female,NaN,57196.0,2,OWN,7000.0,PERSONAL,7.51,0.12,4.0,655.0,Yes,NaN


In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Unnamed: 0                      10000 non-null  int64  
 1   person_age                      8991 non-null   float64
 2   person_gender                   9184 non-null   object 
 3   person_education                9091 non-null   object 
 4   person_income                   10000 non-null  float64
 5   person_emp_exp                  10000 non-null  int64  
 6   person_home_ownership           10000 non-null  object 
 7   loan_amnt                       9553 non-null   float64
 8   loan_intent                     10000 non-null  object 
 9   loan_int_rate                   10000 non-null  float64
 10  loan_percent_income             10000 non-null  float64
 11  cb_person_cred_hist_length      10000 non-null  float64
 12  credit_score                    8

In [16]:
data.isna().sum()

,0
Unnamed: 0,0
person_age,1009
person_gender,816
person_education,909
person_income,0
person_emp_exp,0
person_home_ownership,0
loan_amnt,447
loan_intent,0
loan_int_rate,0


In [17]:
data = drop_data(data=data, columns=['Unnamed: 0', 'previous_loan_defaults_on_file'])
data = drop_missing_data(data=data, columns=['loan_amnt'])

data = simple_impute_data(data=data, columns=['person_age', 'credit_score'], strategy='median')
data = label_data(data=data, columns=['person_gender', 'person_education', 'loan_intent', 'person_home_ownership', 'loan_status'])
data = knn_impute_data(data=data, columns=['person_gender', 'person_education', 'loan_status'], n_neighbors=3, weights='uniform')

In [18]:
data.sample()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
8284,33.0,1.0,0.0,138565.0,10,2,14400.0,5,11.14,0.1,5.0,684.0,0.0


In [19]:
data.isna().sum()

,0
person_age,0
person_gender,0
person_education,0
person_income,0
person_emp_exp,0
person_home_ownership,0
loan_amnt,0
loan_intent,0
loan_int_rate,0
loan_percent_income,0


In [22]:
input_data = data.copy()
input_data = drop_data(data=data, columns=['loan_status'])

target_data = data.copy()['loan_status']

In [23]:
X_train, X_test, Y_train, Y_test = train_test_split(input_data, target_data, test_size=0.2)

### Classification

In [24]:
model = XGBClassifier(max_depth=5, min_child_weight=1, n_estimators=500, n_jobs=-1, learning_rate=0.1)
model.fit(X_train, Y_train)

y_pred = model.predict(X_test)

In [25]:
print("Accuracy:", metrics.accuracy_score(Y_test, y_pred))
print(confusion_matrix(Y_test, y_pred))
print(classification_report(Y_test, y_pred))

Accuracy: 0.8189429618001046
[[1311   54    7]
 [ 119  253    2]
 [ 129   35    1]]
              precision    recall  f1-score   support

         0.0       0.84      0.96      0.89      1372
         1.0       0.74      0.68      0.71       374
         2.0       0.10      0.01      0.01       165

    accuracy                           0.82      1911
   macro avg       0.56      0.55      0.54      1911
weighted avg       0.76      0.82      0.78      1911



In [26]:
classifier = DecisionTreeClassifier(max_depth=3)
classifier = classifier.fit(X_train,Y_train)

y_pred = classifier.predict(X_test)

In [27]:
print("Accuracy:", metrics.accuracy_score(Y_test, y_pred))
print(confusion_matrix(Y_test, y_pred))
print(classification_report(Y_test, y_pred))

Accuracy: 0.8011512297226583
[[1291   81    0]
 [ 134  240    0]
 [ 132   33    0]]
              precision    recall  f1-score   support

         0.0       0.83      0.94      0.88      1372
         1.0       0.68      0.64      0.66       374
         2.0       0.00      0.00      0.00       165

    accuracy                           0.80      1911
   macro avg       0.50      0.53      0.51      1911
weighted avg       0.73      0.80      0.76      1911



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Regression

In [28]:
input_data = data.copy()
input_data = drop_data(data=data, columns=['loan_amnt'])

target_data = data.copy()['loan_amnt']

In [29]:
X_train, X_test, Y_train, Y_test = train_test_split(input_data, target_data, test_size=0.2)

In [30]:
model = XGBRegressor(objective ='reg:linear', colsample_bytree=0.3, learning_rate=0.1, max_depth=7, n_estimators=100)
model.fit(X_train, Y_train)

y_pred = model.predict(X_test)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [12:49:38] WARNING: /workspace/src/objective/regression_obj.cu:282: reg:linear is now deprecated in favor of reg:squarederror.
  bst.update(dtrain, iteration=i, fobj=obj)


In [31]:
print("Mean Absolute Error : " + str(mean_absolute_error(y_pred, Y_test)))
print("R2 Score : " + str(r2_score(y_pred, Y_test)))
print("Mean Squared Error : " + str(mean_squared_error(y_pred, Y_test)))
print("Root Mean Squared Error : " + str(np.sqrt(mean_squared_error(y_pred, Y_test))))

Mean Absolute Error : 1067.960075298571
R2 Score : 0.8955858733468354
Mean Squared Error : 2637531.253580358
Root Mean Squared Error : 1624.0477990442146


In [32]:
LGBM = LGBMRegressor(n_estimators = 50)
LGBM.fit(X_train,Y_train)

y_pred = LGBM.predict(X_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003040 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 921
[LightGBM] [Info] Number of data points in the train set: 7642, number of used features: 12
[LightGBM] [Info] Start training from score 9659.022769


In [33]:
print("Mean Absolute Error : " + str(mean_absolute_error(y_pred, Y_test)))
print("R2 Score : " + str(r2_score(y_pred, Y_test)))
print("Mean Squared Error : " + str(mean_squared_error(y_pred, Y_test)))
print("Root Mean Squared Error : " + str(np.sqrt(mean_squared_error(y_pred, Y_test))))

Mean Absolute Error : 276.535371899743
R2 Score : 0.9936955504625974
Mean Squared Error : 230997.24134516515
Root Mean Squared Error : 480.6217237549351


# Additional helpful things from class


In [34]:
dt = "2025-11-24T13:45:10Z"

date_part = dt[:10]
print(date_part)

2025-11-24


In [35]:
import re

text = "Ana is teaching Python."

matches = re.findall(r"\b[A-Z][a-zA-Z]*\b", text)
print(matches)

['Ana', 'Python']


In [36]:
text = "Contact us at ana@finki.mk or support@example.com"

emails = re.findall(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", text)
print(emails)

['ana@finki.mk', 'support@example.com']


In [37]:
text = "The meeting is on 2025-11-24 at 13:45."

date = re.findall(r"\d{4}-\d{2}-\d{2}", text)
time = re.findall(r"\d{2}:\d{2}", text)
print(date)
print(time)

['2025-11-24']
['13:45']


In [38]:
text = "This is an example (important note) for students."

inside = re.findall(r"\((.*?)\)", text)
print(inside)

['important note']


# Answers to some of your questions


- How to know which encoder?
  - Label Encoder converts each unique category into a number. These numbers have no true order. The most appropriate way to use it is in the target column in classification.
  The problem with using it in feature variables is that it introduces a fake order (not present in the data). For tree-based models this is not that problematic, as it is for Linear Regression, Logistic Regression, KNN, Naive Bayes. In these models performance can be significantly affected.
  - Ordinal Encoder converts categories into numbers, but with meaning. It can use a predefined or natural order. It is natural to use it for ordinal cateogries (for example, clothing sizes, education level etc.). If the categories have no natural order we can use it with our specified order or the 'auto' setting, but still this is introducing some order that can significantly harm these models Linear Regression, Logistic Regression, KNN, Naive Bayes.
  - One-Hot Encoder creates binary columns for each unique category. It is the safest approach for nominal (unordered) variables, and will work the best with models that assume numeric meaning like Linear Regression, Logistic Regression, Neural Networks. However, if the variable has too many categories this will cause a dimensionality problem (too many columns), and for these cases we can use LabelEncoder, however with a model that will not be as affected.

- Can i somehow fill in the target column if it has missing values?
  - No, we will not do this, as it is very risky.

- When to use MICE and when KNN?
  - MICE should be used when the dataset has multiple correlated features, since it can build the regression model well in that instance. MICE will get us statistically sound imputations that preserve distributions. MICE can be used to impute categorical variables, but the standard IterativeImputer we use behaves using regression models only, so it will not be suitable.
  - KNN should be used when the dataset has similar rows, so that it works the best. KNN can be used for both categorical and continuous values, but we need to be careful when imputing with it if we have used One-Hot encoding before and have too many columns. KNN will not do well if we have very imbalanced categories. KNN will also not do well if used for continuous variable, but the dataset is very large, or has many missing values.



